# Mel CNN

## Tanítás

In [7]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from gensim.models import Word2Vec

# --- ÚTVONALAK A COLABON ---
H5_PATH = "/content/spotify_dataset_compressed.h5"
W2V_PATH = "/content/song2vec.model"
# ÚJ MENTÉSI ÚTVONAL: Mel-Only verzió
SAVE_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only.keras"

# ==========================================
# 1. A PINCÉR: DATA GENERATOR (CSAK MEL)
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True, **kwargs):
        super().__init__(**kwargs) # Keras 3 warning eltüntetése!
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_indices = np.sort(batch_indices)

        with h5py.File(self.h5_path, 'r') as hf:
            # CSAK A MEL SPEKTROGRAMOT OLVASSUK BE
            X_mel = hf['spectrograms/mel'][batch_indices]
            uris = hf['ml/track_uri'][batch_indices]

        # Csatorna dimenzió hozzáadása
        X_mel = np.expand_dims(X_mel, axis=-1)

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            if uri in self.w2v_model.wv:
                y_batch[i] = self.w2v_model.wv[uri]

        # Mivel csak 1 bemenet van, simán visszaadjuk (nem kell tuple listába tenni)
        return X_mel, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

# ==========================================
# 2. AZ AGY: MODELL ARCHITEKTÚRA (CSAK MEL)
# ==========================================
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

print("\n🚀 Modell építése (Mel-Only)...")
input_mel = Input(shape=(128, 1280, 1), name='mel_input')

# Csak egyetlen ágat hozunk létre
branch_mel = create_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")

# NINCS CONCATENATE! A Mel ág eredménye egyből megy a Dense rétegekbe:
z = Dense(1024, activation='relu', name='dense_1')(branch_mel)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)
z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = L2NormLayer(name='l2_norm')(output_raw)

# CSAK 1 BEMENET!
model = Model(inputs=input_mel, outputs=output_layer)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=cosine_loss, metrics=['mae'])

# ==========================================
# 3. A STARTPISZTOLY: TANÍTÁS
# ==========================================
print("\n📚 Word2Vec betöltése...")
w2v_model = Word2Vec.load(W2V_PATH)

print("\n⚙️ Generátorok inicializálása...")
train_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='train', batch_size=64)
val_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='val', batch_size=64, shuffle=False)

if os.path.exists(SAVE_PATH):
    print(f"\n💾 Korábbi állapot megtalálva!")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    print("✅ Modell sikeresen betöltve!")
else:
    print(f"\n⚠️ Tiszta lappal indulunk! Nincs mentés ezen a néven: {SAVE_PATH}")

callbacks = [
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print("\n🔥 GPU TANÍTÁS INDÍTÁSA (CSAK MEL SPEKTROGRAM)...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)


🚀 Modell építése (Mel-Only)...

📚 Word2Vec betöltése...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

⚠️ Tiszta lappal indulunk! Nincs mentés ezen a néven: /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only.keras

🔥 GPU TANÍTÁS INDÍTÁSA (CSAK MEL SPEKTROGRAM)...
Epoch 1/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.6751 - mae: 0.1606
Epoch 1: val_loss improved from None to 0.65192, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 701s 2s/step - loss: 0.5566 - mae: 0.1555 - val_loss: 0.6519 - val_mae: 0.1597 - learning_rate: 0.0010
Epoch 2/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.4443 - mae: 0.1508
Epoch 2: val_loss improved from 0.65192 to 0.57666, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only.ker

In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm # Folyamatjelzőhöz: pip install tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only.keras" # Vagy a pontos elérési út
K_VALUES = [10, 20] # Top 10 és Top 20 találatot is vizsgálunk
NUM_TEST_SAMPLES = 500 # Kezdésnek csak 500 dalon teszteljünk, hogy lássuk, lefut-e! (Állítsd None-ra a teljes teszthez)

# Custom rétegek és loss a CNN betöltéséhez
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    """Kiszámolja a metrikákat egy adott K értékre."""
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            np.random.seed(42)
            test_indices = np.random.choice(test_indices, NUM_TEST_SAMPLES, replace=False)
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        # Metrika gyűjtők
        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            # Csak azokat tudjuk tesztelni, amik benne vannak a W2V szótárban
            if uri not in w2v_model.wv:
                continue
                
            # --- GROUND TRUTH KINYERÉSE ---
            pids = track_to_playlists.get(uri, set())
            if not pids: 
                continue # Ha nincs egy listán sem, nem tudjuk tesztelni
                
            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) # Magát a query dalt kivesszük a relevánsak közül
            
            if not relevant_uris:
                continue

            # --- A) BASELINE: Sima Word2Vec predikció ---
            # Bekérjük a legközelebbi szomszédokat a w2v térből
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ ---
            # Kinyerjük az audió feature-öket (hozzáadjuk a batch és channel dimenziókat: 1, X, Y, 1)
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel], verbose=0)[0]
            
            # Megkeressük a prediktált vektorhoz legközelebbi dalokat a W2V szótárban
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            # Kivesszük magát a dalt az ajánlásokból, ha véletlenül azt találná meg
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]
            
            # Előfordulhat, hogy a szűrés miatt 1-el rövidebb a lista, pótoljuk ha kell
            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline (W2V)
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nCNN MODELL (Audio alapján generált vektor):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [01:59<00:00, 555778.95it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése:   0%|          | 0/500 [00:00<?, ?it/s]c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: mel_input
Received: inputs=('Tensor(shape=(1, 128, 1280, 1))',)
  warnings.warn(msg)
Dalok tesztelése: 100%|██████████| 500/500 [01:21<00:00,  6.12it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.60%
  Precision@10: 85.00%
  Recall@10:    0.10%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@10:  53.00%
  Precision@10: 19.26%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  99.40%
  Precision@20: 81.43%
  Recall@20:    0.19%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@20:  64.20%
  Precision@20: 18.87%
  Recall@20:    0.02%


## Kiértékelés

### Releváns uri, word2vec térben keresés

In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only.keras" 
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict(mel, verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*60)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)")
    print("="*60)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)    AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN MODELL -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
    print("\n" + "="*60)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:02<00:00, 543311.26it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [05:25<00:00,  3.07it/s]



VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)

GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.43%
  CNN MODEL (Audio)    AUC: 73.85%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN MODELL -> HR: 20.20% | Prec: 20.20% | Rec: 0.00%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN MODELL -> HR: 42.00% | Prec: 18.76% | Rec: 0.01%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN MODELL -> HR: 52.40% | Prec: 18.71% | Rec: 0.01%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN MODELL -> HR: 62.60% | Prec: 18.65% | Rec: 0.03%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN MODELL -> HR: 73.30% | Prec: 18.73% | Rec: 0.07%

--- TOP 100 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 70.46% | Rec: 0.93%
  CNN MODELL -> HR: 78.10% | Prec: 18.55% | Rec: 0.14%

--- TOP 200 AJÁNLÁS ---
  BASELINE -> HR: 99.90% | Prec: 64

### Releváns uri, hibrid téren (valós környezet)

In [3]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score
import copy  # Hozzáadva a hibrid modell klónozásához!

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only.keras" 
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_mel.npy" # A létrehozott hibrid mátrixod

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n[+] HIBRID vektortér betöltése a memóriába...")
    # Klónozzuk a w2v modellt, hogy a szótár (vocab) megmaradjon
    hybrid_w2v = copy.deepcopy(w2v_model)
    
    # Betöltjük a numpy mátrixot
    hybrid_matrix = np.load(HYBRID_MATRIX_PATH)
    
    # FELÜLÍRJUK a klónozott modell vektorait a hibrid mátrixszal!
    hybrid_w2v.wv.vectors = hybrid_matrix
    
    # KRITIKUS LÉPÉS: Újraszámoljuk a normákat a cosine similarity-hez
    hybrid_w2v.wv.fill_norms(force=True) 
    print("Hibrid vektortér sikeresen integrálva!")

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár (kibővítve a hibriddel)
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "hybrid":   {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict(mel, verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- C) HIBRID MODELL ---
            hybrid_similars = hybrid_w2v.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            hybrid_recs = [u for u, score in hybrid_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))
            results["hybrid"]["auc"].append(calculate_auc(pred_vector, relevant_uris, hybrid_w2v))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)
                
                # Hybrid
                hr_h, prec_h, rec_h = calculate_metrics(hybrid_recs, relevant_uris, k)
                results["hybrid"][k]["hr"].append(hr_h)
                results["hybrid"][k]["prec"].append(prec_h)
                results["hybrid"][k]["rec"].append(rec_h)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*70)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆")
    print("="*70)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)   AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")
    print(f"  HIBRID MODELL       AUC: {np.mean(results['hybrid']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN      -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
        print(f"  HIBRID   -> HR: {np.mean(results['hybrid'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['hybrid'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['hybrid'][k]['rec'])*100:.2f}%")
    print("\n" + "="*70)

if __name__ == "__main__":
    main()

1. Modellek betöltése...

[+] HIBRID vektortér betöltése a memóriába...
Hibrid vektortér sikeresen integrálva!

2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:55<00:00, 378523.26it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [09:58<00:00,  1.67it/s]



VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆

GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.45%
  CNN MODEL (Audio)   AUC: 73.83%
  HIBRID MODELL       AUC: 57.98%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN      -> HR: 20.20% | Prec: 20.20% | Rec: 0.00%
  HIBRID   -> HR: 55.10% | Prec: 55.10% | Rec: 0.01%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN      -> HR: 42.00% | Prec: 18.76% | Rec: 0.01%
  HIBRID   -> HR: 86.60% | Prec: 55.60% | Rec: 0.03%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN      -> HR: 52.40% | Prec: 18.71% | Rec: 0.01%
  HIBRID   -> HR: 91.80% | Prec: 55.12% | Rec: 0.06%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN      -> HR: 62.60% | Prec: 18.65% | Rec: 0.03%
  HIBRID   -> HR: 94.60% | Prec: 54.37% | Rec: 0.12%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN    

### Szekvenciális hibrid téren

In [4]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_mel.npy"
TEST_PIDS_PATH = "../Models/test_pids.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists[::10]): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = pl[:-1]
    target_idx = pl[-1]
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 99,180 listán...


100%|██████████| 9918/9918 [05:42<00:00, 28.99it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.26%
  MAP@1:            0.2621%
  NDCG@1:           0.2621%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.70%
  MAP@5:            0.4173%
  NDCG@5:           0.4865%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  1.09%
  MAP@10:            0.4657%
  NDCG@10:           0.6096%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  1.65%
  MAP@20:            0.5018%
  NDCG@20:           0.7485%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  3.33%
  MAP@50:            0.5525%
  NDCG@50:           1.0761%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  5.48%
  MAP@100:            0.5834%
  NDCG@100:           1.4261%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  8.79%
  MAP@200:            0.6069%
  NDCG@200:           1.8879%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  15.35%
  MAP@500:            0.6274%
  NDCG@500:           2.6722%



### Szekvenciális, tartalom alapú

In [1]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/ae_vectors_closed_world_mel_only.npy"
TEST_PIDS_PATH = "../Models/test_pids_ae.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = [idx - 1 for idx in pl[:-1]]
    target_idx = pl[-1] - 1
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 88,316 listán...


100%|██████████| 88316/88316 [03:31<00:00, 416.60it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.02%
  MAP@1:            0.0204%
  NDCG@1:           0.0204%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.08%
  MAP@5:            0.0406%
  NDCG@5:           0.0514%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  0.18%
  MAP@10:            0.0529%
  NDCG@10:           0.0820%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  0.33%
  MAP@20:            0.0631%
  NDCG@20:           0.1192%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  0.79%
  MAP@50:            0.0769%
  NDCG@50:           0.2091%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  1.45%
  MAP@100:            0.0861%
  NDCG@100:           0.3161%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  2.68%
  MAP@200:            0.0946%
  NDCG@200:           0.4869%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  5.92%
  MAP@500:            0.1046%
  NDCG@500:           0.8737%

